# House price prediction - linear regression

In [ ]:
import pandas as pd

# this is a relative path
df = pd.read_csv('../../data/house_prices.csv')



waterfront
0    4567
1      33
Name: count, dtype: int64

In [ ]:
# Checking how many are waterfront vs. how many are not
df["waterfront"].value_counts()

waterfront
0    4567
1      33
Name: count, dtype: int64

In [ ]:
# Objects are our categorical features
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [16]:
import duckdb as db

query = db.query (
    """
    SELECT
        AVG(price) AS avg_price,
        condition
    FROM
        df
    GROUP BY
        2
    """
).df()

query

,avg_price,condition
0,637041.322258,5
1,550111.516394,3
2,324373.750000,2
3,306633.333333,1
4,533647.286072,4


In [23]:
df_cleaned = df.drop(["country", "statezip", "city", "street", "date"], axis="columns")
df_cleaned["yr_renovated"] = df.apply(lambda x: x["yr_built"] if x["yr_renovated"] == 0 else x["yr_renovated"], axis= 1)
df_cleaned.head(5)

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,1921
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,1966
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,1963
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992


In [21]:
X, y = df_cleaned.drop("price", axis=1), df_cleaned["price"]

y.head()

0     313000.0
1    2384000.0
2     342000.0
3     420000.0
4     550000.0
Name: price, dtype: float64

In [28]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

X_train.shape, X_test.shape

((3082, 12), (1518, 12))

In [ ]:
# scale dataset
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

scaled_X_train = scaler.transform(X_train)
scaled_X_test = scaler.transform(X_test)



(np.float64(-1.6267901295286201e-16), np.float64(1.0117511262934307))

In [33]:
# predict house price

from sklearn.linear_model import LinearRegression, ElasticNetCV, RidgeCV

model = LinearRegression()

def predict_house_price(model):
    model.fit(scaled_X_train, y_train)
    y_pred = model.predict(scaled_X_test)

    return y_pred

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def compute_metrics(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    return {"MAE": mae, "MSE": mse, "RMSE": rmse}

y_pred_linear_regression = predict_house_price(LinearRegression())
y_pred_ridge = predict_house_price(RidgeCV())
y_pred_elasticnet = predict_house_price(ElasticNetCV())





{'MAE': 259611.936785628,
 'MSE': 664290227641.4569,
 'RMSE': np.float64(815040.0159755697)}

In [39]:
# compute metrics for LR
compute_metrics(y_test, y_pred_linear_regression)

{'MAE': 191902.15686061102,
 'MSE': 621380924636.5221,
 'RMSE': np.float64(788277.1876925795)}

In [40]:
# Compute metrics for RidgeCV
compute_metrics(y_test, y_pred_ridge)

{'MAE': 191756.38824126765,
 'MSE': 621240464122.7036,
 'RMSE': np.float64(788188.0893052772)}

In [41]:
# Compute metrics for elastic net CV
compute_metrics(y_test, y_pred_elasticnet)

{'MAE': 259611.936785628,
 'MSE': 664290227641.4569,
 'RMSE': np.float64(815040.0159755697)}